# V7 seaborn 관계·분포 — 그림은 몇 개로 만들었는지 말하지 않는다 · 실습 (T7)

> **6단계 프레임: ⑥쓰임** — 세트피스는 **⑤확인**(그림의 숫자마다 **몇 개로 만들었는지**를 묻습니다).

**이 노트북의 목표**

1. 상관표를 만들고 히트맵으로 그린다 — 그리고 **어떤 열이 들어갔는지** 눈으로 확인한다
2. **세트피스** ⭐ — 히트맵의 세 칸이 **각각 몇 줄로 만들어졌는지** 손으로 센다
3. 고치는 **두 길**을 다 걸어 보고, 각각 무엇을 얻고 무엇을 잃는지 표로 따라간다
4. 두 무리를 견주는 질문에 맞는 그림을 고르고, 상자의 **다섯 자리**를 손으로 읽는다
5. 색에도 자가 있다는 것을 확인하고, **오류 다섯 가지**를 읽는다

**빈칸은 `___` 입니다.** 총 **7개**.

## Part A. 준비 — V4의 그 표에 강수량을 더한다

**V3 §12 · V4 §2 · V6 §8**이 세 번 "강수량은 V7에서 더한다"고 미뤄 둔 열을 오늘 붙입니다.
비가 온 날에만 값이 있고 **안 온 날은 빈 칸**입니다.

🔴 **PD8에서는 이 빈 칸을 채웠습니다** — 기상 자료의 빈 강수량은 모르는 값이 아니라 **0mm**니까요.
오늘은 **일부러 그대로 두고** `corr`가 그것을 어떻게 다루는지 먼저 봅니다. 이 셀은 **그대로 실행만 하세요.**

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

rentals = [14230, 15010, 13120, 13540, 6880, 12760, 13980, 16420, 15870, 13310,
           13650, 9240, 14020, 14890, 16060, 5120, 7340, 10210, 4050, 8760,
           12480, 5430, 5970, 11390, 9880, 11390, 4610, 7890, 13240, 13880]
temp = [24.1, 23.5, 22.8, 24.6, 21.2, 23.9, 25.3, 26.0, 26.4, 25.1,
        24.7, 22.9, 25.8, 26.6, 27.2, 23.1, 22.4, 24.0, 21.6, 23.3,
        26.1, 22.0, 21.8, 25.4, 24.2, 26.8, 22.6, 23.0, 27.5, 27.9]
week = ["토", "일", "월", "화", "수", "목", "금"]          # 2024년 6월 1일은 토요일
rain_mm = {5: 8.5, 12: 3.2, 16: 12.4, 17: 5.6, 19: 22.8, 20: 3.1,
           22: 15.2, 23: 9.8, 25: 1.2, 27: 18.5, 28: 6.4}   # 비가 온 날만 적혀 있다

df = pd.DataFrame({
    "날짜": list(range(1, 31)),
    "요일": [week[i % 7] for i in range(30)],
    "대여건수": rentals,
    "평균기온": temp,
    "강수량": [rain_mm.get(d, None) for d in range(1, 31)],   # 없는 날짜면 None -> 빈 칸
})

print("모양:", df.shape)
print(df.head(3))
print("seaborn 판:", sns.__version__)

## Part B. 관계를 보는 그림은 점이었다 (V2)

두 값이 어떻게 같이 움직이는지 보는 그림은 **점**입니다. 빈칸 없음.

In [ ]:
figB, axB = plt.subplots(figsize=(6, 4))
sns.scatterplot(data=df, x="평균기온", y="대여건수", ax=axB)
axB.set_title("Rentals vs temperature")
axB.set_xlabel("Temperature")          # 안 붙이면 한글 열 이름이 자동으로 붙어 네모가 된다
axB.set_ylabel("Rentals")
figB.tight_layout()
plt.show()

숫자 열이 셋이니 짝을 지으면 그림이 **세 장**입니다. 열이 넷이면 여섯 장, 다섯이면 열 장이지요.
그래서 관계의 세기를 **숫자 하나**로 줄여 표로 만듭니다 — 그것이 **상관계수**입니다.

## Part C. 상관표 한 장으로 — 빈칸 1

`df.corr()`를 그냥 부르면 `요일` 열이 글자라서 멈춥니다.
**숫자 열만 보라고 일러 주는 인자**를 빈칸에 적으세요(설명서 §3.1).

In [ ]:
wide = df.corr(___=True)          # ✍️ 빈칸 1 — 숫자 열만 보라고 일러 주는 인자
print(wide.round(2))
print()
print("들어간 열:", list(wide.columns))

표가 **넷 곱하기 넷**으로 나왔습니다. `날짜`가 끼어들었기 때문입니다 —
`날짜`는 1부터 30까지의 **순번**이지 잰 값이 아닌데, 그 인자는 **자료형만** 보고 **뜻은 보지 않습니다.**

그래서 볼 열을 **내가 적습니다.**

In [ ]:
cols = ["대여건수", "평균기온", "강수량"]
c = df[cols].corr()
print(c.round(2))

## Part D. 세트피스 ⭐ — 히트맵을 그린다

아홉 개 숫자를 색으로 칠하면 한눈에 들어옵니다.
축 이름표를 영어로 바꿔야 하는데, **V4의 규칙대로 순서를 먼저 확인**하고 나서 갈아 끼웁니다. 빈칸 없음.

In [ ]:
print("열 순서:", list(c.columns))          # 이름표를 갈아 끼우기 전에 순서부터 본다

labels = ["Rentals", "Temp", "Rain"]

figD, axD = plt.subplots(figsize=(5.2, 4.2))
sns.heatmap(c, annot=True, xticklabels=labels, yticklabels=labels, ax=axD)
axD.set_title("Correlation of three columns")
figD.tight_layout()
plt.show()

print("칸에 찍힌 글자:", [t.get_text() for t in axD.texts])

오류도 경고도 없습니다. 색은 곱고 숫자도 아홉 개가 다 찍혔습니다.

## Part E. 손으로 먼저 — 빈칸 2

**아래 셀을 실행하기 전에** 위 Part A 의 `rain_mm` 을 눈으로 세어 보세요.
표는 서른 날인데, 그 가운데 **강수량이 비어 있는 날**은 며칠일까요?

그리고 히트맵의 **세 칸**은 각각 **몇 날**로 만들어졌을지도 손으로 적어 두세요.

| 칸 | 값 | 쓰인 날 수 |
|---|---|---|
| 대여건수 ↔ 평균기온 | 0.77 | ? |
| 대여건수 ↔ 강수량 | -0.95 | ? |
| 평균기온 ↔ 강수량 | -0.57 | ? |

In [ ]:
missing_guess = ___          # ✍️ 빈칸 2 — 강수량이 비어 있는 날은 며칠일까요

print("맞았나:", missing_guess + int(df["강수량"].count()) == len(df))

In [ ]:
pairs = [["대여건수", "평균기온"], ["대여건수", "강수량"], ["평균기온", "강수량"]]

for p in pairs:
    both = (df[p[0]].notna()) & (df[p[1]].notna())          # PD3 의 괄호 규칙 그대로
    print(p[0], "-", p[1], "->", int(both.sum()), "날")

🔴 **같은 그림 안의 세 칸이 서로 다른 날 수로 만들어졌습니다.**
그런데 그림에는 그 숫자가 **하나도 나오지 않습니다.**

두 열의 상관계수를 재려면 **한 줄에 두 값이 다 있어야** 합니다.
6월 1일은 대여건수는 있는데 강수량이 없으니, 그 줄은 **아무 말 없이 빠집니다.**

## Part F. 고치는 길 ① — 짝을 맞춘다 (빈칸 3)

칸마다 자가 다른 것이 문제라면, **모든 칸을 같은 줄들로** 만들면 됩니다.
PD4에서 배운 **빈 칸이 든 줄을 통째로 버리는 메서드**를 빈칸에 적으세요.

In [ ]:
paired = df[cols].___()          # ✍️ 빈칸 3 — 빈 칸이 든 줄을 버린다
print("모양:", paired.shape)
print(paired.corr().round(2))

세 칸이 전부 같은 줄들로 계산되어 이제 **칸끼리 견줄 수** 있습니다.
**그 대신 왼쪽 위 칸의 숫자가 바뀌었습니다** — 위 Part C 의 표와 견줘 보세요.
버린 줄만큼 데이터가 사라졌고, 남은 숫자의 **뜻도 바뀌었습니다.**

## Part G. 고치는 길 ② — 빈 칸의 뜻을 적어 넣는다 (빈칸 4)

빈 칸이 **모르는 값**이면 버리는 수밖에 없습니다. 그런데 이 빈 칸은 **모르는 값이 아닙니다** —
비가 **안 왔다**는 뜻이고, 그날 강수량은 **0mm** 입니다. **PD8이 이 열에 쓴 그 메서드**를 빈칸에 적으세요.

In [ ]:
filled = df[cols].___(0)          # ✍️ 빈칸 4 — 빈 칸에 0 을 적어 넣는다
print("모양:", filled.shape)
print(filled.corr().round(2))

**한 줄도 버리지 않았습니다.** 세 칸이 전부 서른 날입니다.
그 대신 이번에는 **아래 두 칸의 숫자가 바뀌었습니다** — 비가 안 온 날들이 계산에 들어왔기 때문입니다.

🔴 **셋 다 틀리지 않았습니다. 서로 다른 질문의 답일 뿐입니다**(PD8의 그 문장).
이 표에서는 빈 강수량이 **0mm** 라 길 ②가 맞습니다. **PD4의 나이 열이었다면 반대**였습니다 —
0으로 채우면 **0살 승객**이 생기니까요. **같은 도구, 반대 판단.**

## Part H. 질문을 다시 — 비 온 날과 안 온 날

우리가 알고 싶었던 것은 **"비가 오면 대여가 주는가"** 였습니다.
그런데 상관계수는 **비가 셀수록 어떻게 되는지**를 잴 뿐, **비 온 날이 안 온 날보다 얼마나 적은지**는 말해 주지 않습니다.
이 질문은 **두 값의 관계**가 아니라 **두 무리의 비교**입니다. 빈칸 없음.

아래 셀은 그림을 둘 그립니다 — V2에서 **눈으로만 짚고 넘어갔던** 아래쪽 점들을 색으로 가르고,
점이 겹쳐 보이니 무리마다 **분포를 요약한 상자**로도 세웁니다.

In [ ]:
df["비"] = df["강수량"].notna().map({True: "Rain", False: "Dry"})   # 무리는 한 열의 값이어야 한다(V6)
print(df["비"].value_counts())

figH, axH = plt.subplots(figsize=(6, 4))
sns.scatterplot(data=df, x="평균기온", y="대여건수", hue="비",
                hue_order=["Dry", "Rain"], ax=axH)
axH.set_title("Rentals vs temperature, split by rain")
axH.set_xlabel("Temperature")
axH.set_ylabel("Rentals")
axH.legend(title="Rain today?")
figH.tight_layout()
plt.show()

figH2, axH2 = plt.subplots(figsize=(5.6, 4.2))
sns.boxplot(data=df, x="비", y="대여건수", order=["Dry", "Rain"], ax=axH2)
axH2.set_title("Rentals: dry days vs rainy days")
axH2.set_xlabel("Rain today?")
axH2.set_ylabel("Rentals")
figH2.tight_layout()
plt.show()

print(df.groupby("비")["대여건수"].mean().round(1))

🔴 이 그림은 **서른 날을 하나도 버리지 않았습니다.** 상관계수 한 칸으로 답할 수 없던 질문이 상자 둘로 답해집니다.

## Part I. 상자 하나 읽기 — 빈칸 5

상자 안의 굵은 선은 **평균이 아니라 중앙값**입니다.
아래에서 두 무리의 대여건수를 **날짜 순서 그대로** 찍습니다.
**비 온 날 쪽을 손으로 줄 세워** 한가운데 오는 값을 찾으세요(열한 개니 **여섯째**입니다).

In [ ]:
rain_days = list(df[df["비"] == "Rain"]["대여건수"])
dry_days = list(df[df["비"] == "Dry"]["대여건수"])

print("비 온 날(날짜 순):", rain_days)
print("맑은 날(날짜 순):", dry_days)
print("몇 날인가:", len(rain_days), "/", len(dry_days))

In [ ]:
med_rain = ___          # ✍️ 빈칸 5 — 비 온 날을 줄 세웠을 때 한가운데 오는 값

rain_sorted = sorted(rain_days)
below = sum(1 for v in rain_days if v < med_rain)
above = sum(1 for v in rain_days if v > med_rain)
print("아래에 몇 개 · 위에 몇 개:", below, above, "| 같으면 한가운데:", below == above)

## Part J. 수염 밖의 점 — 빈칸 6

수염은 상자에서 **IQR의 1.5배** 안에 드는 값 가운데 **가장 먼 실제 값**까지만 뻗습니다.
그 밖에 남은 값은 **점 하나로 따로** 찍힙니다. 맑은 날 무리의 아래 울타리를 계산해 보고,
위 Part I 가 찍어 준 맑은 날 목록에서 **그보다 낮은 값을 손으로 찾아** 빈칸에 적으세요.

In [ ]:
dry_stats = df[df["비"] == "Dry"]["대여건수"].describe()
q1 = float(dry_stats["25%"])
q3 = float(dry_stats["75%"])
fence_low = q1 - 1.5 * (q3 - q1)

print("Q1:", q1, "| Q3:", q3, "| IQR:", q3 - q1)
print("아래 울타리:", fence_low)

In [ ]:
outlier = ___          # ✍️ 빈칸 6 — 울타리보다 낮아 점으로 따로 찍히는 값

print("울타리보다 낮은가:", outlier < fence_low)
print("울타리 밖에 남은 값이 이것 하나뿐인가:", [v for v in dry_days if v < fence_low] == [outlier])

In [ ]:
print(df.groupby("비")["대여건수"].describe()[["min", "25%", "50%", "75%", "max"]])

🔴 두 무리의 다섯 자리를 견줘 보세요. **맑은 날의 가장 작은 값이 비 온 날의 가장 큰 값보다 큽니다** —
서른 날 가운데 **단 하루도** 예외가 없습니다. 상관계수 한 칸은 이 사실을 말해 주지 못했습니다.

> 그래도 상자에 **없는 것이 하나** 있습니다 — **몇 개로 만들었는가.**
> 왼쪽이 몇 날이고 오른쪽이 몇 날인지는 **화면 어디에도 없습니다.** 그것이 **V8**입니다.

## Part K. 색에도 자가 있다 — 빈칸 7

Part D 의 그림을 다시 보세요. 기본 팔레트는 **값이 클수록 밝습니다** —
그래서 "제일 진한 칸이 제일 센 관계"라는 읽기가 틀립니다(0.77도 센데 밝은 쪽입니다).
게다가 자의 양끝을 **데이터가** 정합니다.

**색의 자를 통째로 고르는 인자**를 빈칸에 적으세요(색 하나를 고르는 `color=`가 아닙니다).

In [ ]:
figK, axK = plt.subplots(figsize=(5.2, 4.2))
sns.heatmap(c, annot=True, ___="coolwarm", vmin=-1, vmax=1,          # ✍️ 빈칸 7
            xticklabels=labels, yticklabels=labels, ax=axK)
axK.set_title("Correlation, fixed scale -1 to 1")
figK.tight_layout()
plt.show()

print("색 자의 양끝:", axK.collections[0].get_clim())   # 그림이 실제로 쓴 자를 되묻는다

숫자는 하나도 바뀌지 않았습니다. **바뀐 것은 색뿐입니다.**
`vmin`·`vmax`로 양끝을 못 박았으니 이제 **어느 표를 그려도 같은 색이 같은 값**입니다.

> 🔴 딥러닝 D4a·D4b 노트북이 `imshow(..., vmin=0, vmax=1)`·`imshow(..., vmin=-1, vmax=1)`로 그리는 것이 같은 이야기입니다.

## Part L. 🔹심화 — `pairplot` 과 V5의 두 반달

`pairplot`은 숫자 열을 **모든 짝**으로 산점도를 그립니다. 히트맵이 감춘 숫자가 **점의 개수**로 보입니다.
🔴 **Figure를 자기가 만들어 `ax=`를 받지 못합니다** — V6이 "이 트랙은 axes-level만 쓴다"고 한 원칙의 **하나뿐인 예외**입니다. 빈칸 없음.

In [ ]:
eng = pd.DataFrame({"Rentals": df["대여건수"], "Temp": df["평균기온"], "Rain": df["강수량"]})

g = sns.pairplot(eng)          # g.axes[줄][칸] 이 격자의 한 칸, .collections[0] 이 그 칸의 점 뭉치
print("Rentals-Temp 칸의 점:", len(g.axes[0][1].collections[0].get_offsets()))
print("Rentals-Rain 칸의 점:", len(g.axes[0][2].collections[0].get_offsets()))
plt.show()

front = df.head(15)
back = df.tail(15)

print("앞 보름 평균:", round(front["대여건수"].mean(), 1), "| 비 온 날:", int((front["비"] == "Rain").sum()))
print("뒤 보름 평균:", round(back["대여건수"].mean(), 1), "| 비 온 날:", int((back["비"] == "Rain").sum()))
print("맑은 날만 앞 보름:", round(front[front["비"] == "Dry"]["대여건수"].mean(), 1))
print("맑은 날만 뒤 보름:", round(back[back["비"] == "Dry"]["대여건수"].mean(), 1))

V5에서 견준 두 반달의 차이가 **비 온 날 수**와 함께 움직입니다.
🔴 그렇다고 **"비가 원인이다"** 라고 말할 수는 없습니다 — 그림은 **같이 움직이는 것**만 보여 줍니다.

## Part M. 오류 읽기 다섯 가지

아래 다섯 줄을 **하나씩 주석을 풀어** 실행하고 오류 이름과 문구를 읽으세요.
읽었으면 다시 주석으로 막아야 다음 셀이 실행됩니다. 빈칸 없음.

In [ ]:
figM, axM = plt.subplots()

# 1) df.corr()                                             # 글자 열이 섞여 있다
# 2) df.corr(numeric=True)                                 # 인자 이름을 잘못 적었다
# 3) sns.heatmap(df, ax=axM)                               # 상관표가 아니라 원본 표를 넘겼다
# 4) sns.boxplot(data=df, x="비ㅁ", y="대여건수", ax=axM)     # 그런 열 이름이 없다
# 5) sns.heatmap(c, annot=True, color="coolwarm", ax=axM)  # 색 하나와 색의 자를 헷갈렸다

print("다섯 줄을 하나씩 풀어 실행해 보세요. 읽었으면 다시 주석 처리합니다.")
plt.close(figM)

## Part N. `assert` 자가 채점

전부 맞으면 마지막 문구가 나옵니다. 하나라도 틀리면 그 줄에서 멈춥니다.

In [ ]:
assert "날짜" in list(wide.columns) and len(wide.columns) == len(c.columns) + 1
assert missing_guess + int(df["강수량"].count()) == len(df)
assert len(paired) == int(df["강수량"].count()) and len(paired) < len(df)
assert round(float(paired.corr().loc["대여건수", "평균기온"]), 2) != round(float(c.loc["대여건수", "평균기온"]), 2)
assert len(filled) == len(df)
assert round(float(filled.corr().loc["대여건수", "강수량"]), 2) != round(float(c.loc["대여건수", "강수량"]), 2)
assert round(float(filled.corr().loc["대여건수", "평균기온"]), 2) == round(float(c.loc["대여건수", "평균기온"]), 2)
assert med_rain in rain_days and sorted(rain_days).index(med_rain) * 2 + 1 == len(rain_days)
assert med_rain != round(float(df[df["비"] == "Rain"]["대여건수"].mean()), 1)
assert [v for v in dry_days if v < fence_low] == [outlier]
assert tuple(axK.collections[0].get_clim()) == (-1.0, 1.0)
assert min(dry_days) > max(rain_days)

print("전부 맞았습니다. 그림의 숫자를 믿기 전에 그것이 몇 줄로 만들어졌는지 먼저 셉니다.")